In [0]:
#Validate Required Fields — Create a function that accepts a transaction record and a set of required fields, then returns whether the record is valid along with the missing-field information.
'''{"transaction_id": "T001", "customer_id": "C101", "amount": 125.50, "currency": "USD"}
Assumptions- 1. transaction_id,customer_id, amount are required fields
             2. record can accept 2 additional optional fields - is_refund(bool) &  source(str)
             3. Only few standard currencies are accepted'''
#import
from decimal import Decimal, InvalidOperation
from typing import Final, Optional 
import re
from dataclasses import dataclass
import math
#global variables
REQUIRED_FIELDS:Final[frozenset[str]]=frozenset({"transaction_id","customer_id", "amount" })
#VALID_CUURENCIES:Final(frozenset)={"USD", "POUND", "EURO"}
VALID_TRANID_FORMAT:Final[re.Pattern[str]]=re.compile(r"T\d{3}")
VALID_CUSTID_FORMAT:Final[re.Pattern[str]]=re.compile(r"C\d{3}")
#data class 
@dataclass
class record:
    transaction_id: str|None
    customer_id:str|None
    amount:Decimal|None
    currency:Optional[str]=None
    is_refund:Optional[bool]=None
    source:Optional[str]=None

#validaion of each field
def validate_record(in_record:object)->record:
    #validate if record is empty or non string value
    if not isinstance(in_record, dict):
        raise TypeError(f"record expected to be in directory format"
                        f"received {type(in_record).__name__}")
    record_keys=list(in_record)
    if len(record_keys)==0:
        raise ValueError(f"record is empty")
    #validate record length matches with number of manadatory fields
    if len(record_keys)<len(REQUIRED_FIELDS):
        raise ValueError(f"record has missing required fields"
                         f"minimum expected fields={len(REQUIRED_FIELDS)}"
                         f"received={len(record_keys)}")     
    tran_id=record_keys[0]
    customer_id=record_keys[1]
    amount=record_keys[2]  
    #validate transaction id
    cleaned_tran_id=tran_id.strip()
    if not cleaned_tran_id:
        raise ValueError(f"required field transaction_id is missing")
    if not VALID_TRANID_FORMAT.fullmatch(cleaned_tran_id):
        raise ValueError(f"transaction id has invalid format")
    #validate customer id
    cleaned_cust_id=customer_id.strip()
    if not cleaned_cust_id:
        raise ValueError("required field customer id is missing")
    if not VALID_CUSTID_FORMAT.fullmatch(cleaned_cust_id):
        raise ValueError(f"invalid customer id format")
    #validate amount     
    try:
        Decimal(amount)
    except (InvalidOperation, TypeError):
        f"amount expected to be decimal value received {Type(amount).__name__}"
    
    if amount<0 or not math.isfinite(amount):
        raise ValueError(f"amount must be positive & finite number")

    #return record
    return record(
        transaction_id=cleaned_tran_id,
        customer_id=cleaned_cust_id,
        amount=amount,
        currency='USD',
        is_refund=False,
        source=None
    ) 
#main function
if __name__=="__main__":
    record_list=[
        {"transaction_id": "T001", "customer_id": "C101", "amount": 125.50, "currency": "USD"},
        {"transaction_id": "T002", "amount": 80.00, "currency": "USD"},
        {"transaction_id": "T009", "customer_id": "C109", "amount": 300.00, "currency": "USD", "source": "POS"}
    ]
    for rec in record_list:
        print(validate_record(rec))

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-6910002903081658>, line 82
     76 record_list=[
     77     {"transaction_id": "T001", "customer_id": "C101", "amount": 125.50, "currency": "USD"},
     78     {"transaction_id": "T002", "amount": 80.00, "currency": "USD"},
     79     {"transaction_id": "T009", "customer_id": "C109", "amount": 300.00, "currency": "USD", "source": "POS"}
     80 ]
     81 for rec in record_list:
---> 82     print(validate_record(rec))

File <command-6910002903081658>, line 49, in validate_record(in_record)
     47     raise ValueError(f"required field transaction_id is missing")
     48 if not VALID_TRANID_FORMAT.fullmatch(cleaned_tran_id):
---> 49     raise ValueError(f"transaction id has invalid format")
     50 #validate customer id
     51 cleaned_cust_id=customer_id.strip()

ValueError: transaction id has invalid format

In [0]:
#Validate Required Fields — Create a function that accepts a transaction record and a set of required fields, then returns whether the record is valid along with the missing-field information.
import logging
from collections.abc import Mapping, Set
from dataclasses import dataclass
from typing import Final, Any

Logger=logging.getLogger(__name__) 
DEFAULT_REQUIRED_FIELDS: Final[frozenset[str]] = frozenset(
    {"transaction_id", "customer_id", "amount"}
)
@dataclass(frozen=True)
class RequiredFieldValidation:
    """Result of checking whether required transaction fields are populated."""
    is_valid:bool
    missing_fields:tuple[str,...]

def _is_missing(value:Any)->bool:
    """Return True only for null values or whitespace-only strings."""
    return value is None or (isinstance(value, str) and not value.strip())

def validate_required_fields(
    transactions: Mapping[str, Any],
    required_fields:Set[str]=DEFAULT_REQUIRED_FIELDS,
)->RequiredFieldValidation:
    """
    Validate presence of required fields in one transaction record.

    A field is missing when it is absent, has a value of None, or contains
    an empty/whitespace-only string. Numeric zero is considered present.

    Raises:
        TypeError: If transaction is not a mapping or required fields contain
            non-string values.
        ValueError: If any configured required field is blank.
    """
    if not isinstance(transactions, Mapping):
        raise TypeError("transaction must be mapping"
                        f"received {type(transactions).__name__}")
    if any(not isinstance(field, str) for field in required_fields):
        raise TypeError("required fields contains strings")
    if any(not field.strip() for field in required_fields):
        raise ValueError("required field cannot be empty")
    missing_fields=tuple(
        sorted(
            field for field in required_fields
            if field not in transactions or _is_missing(transactions[field])
        )
    )
    result=RequiredFieldValidation(
        is_valid=not missing_fields,
        missing_fields=missing_fields,
    )
    Logger.debug(
        "validated required transaction fields: is_valid=%s, missing_fields=%s",
        result.is_valid,
        result.missing_fields
    )
    return result

if __name__=="__main__":
    records = [
        {
            "transaction_id": "T001",
            "customer_id": "C101",
            "amount": 125.50,
            "currency": "USD",
        },
        {
            "transaction_id": "T002",
            "amount": 80.00,
            "currency": "USD",
        },
        {
            "transaction_id": "T009",
            "customer_id": "C109",
            "amount": 300.00,
            "currency": "USD",
            "source": "POS",
        },
    ]
    for transaction in records:
        result=validate_required_fields(transaction)
        print(result)

RequiredFieldValidation(is_valid=True, missing_fields=())
RequiredFieldValidation(is_valid=False, missing_fields=('customer_id',))
RequiredFieldValidation(is_valid=True, missing_fields=())


In [0]:
#Parse Monetary Values — Build a function that converts incoming monetary values such as "1,250.45", "$99.00", " 500 ", integers, floats, empty strings, and malformed values into a consistent numeric representation.
#import
from decimal import Decimal, InvalidOperation 
from typing import Final
import re
import math
#global variables
SUPPORTED_CURRENCIES:Final[frozenset[str]]=frozenset({'USD', '¥', 'INR', '$'})
CURRENCY_SYMBOL_PATTERN=re.compile(r"[^\d\s]")
#Validation & normalization
def currency_validation(currency:object)->Decimal:
    #incoming value is string or not
    if not isinstance(currency, (str)):
        raise TypeError(f"currency expected to be in string format received {type(currency).__name__}")
    #incoming value is empty
    cleaned_currency=currency.strip()
    if not cleaned_currency:
        raise ValueError(f"currency cannot be empty") 
    #incoming value has supported currency
    currency_symbol=CURRENCY_SYMBOL_PATTERN.findall(cleaned_currency)
    if currency_symbol not in SUPPORTED_CURRENCIES and \
         (currency_symbol not in('(' , ')')):
        raise ValueError(f"unsupported currency. supported currencies are {SUPPORTED_CURRENCIES}")
    #incoming value has commas then remove them
    try:
        cleaned_currency=Decimal(cleaned_currency.replace(",",""))
        #cleaned_currency=Decimal(cleaned_currency.replace("_",""))
    except:
        pass
    #remove leading zeros
    '''try:
        cleaned_currency=Decimal(cleaned_currency.replace("0",""))
    except:
        pass'''
    #incoming value is infinity 
    if math.isinf(cleaned_currency):
        raise ValueError("currencies cannot be infinite") 
    #incoming value is positive decimal 
    try:
        cleaned_currency=Decimal(cleaned_currency)
    except(InvalidOperation, ValueError):
        raise ValueError(f"not a decimal value")         
    #incoming value is negative decimal
    if currency_symbol in ('(',')'):
        cleaned_currency=cleaned_currency*-1 
    return cleaned_currency

#main function
if __name__=="__main__":
    currency_list=[
        "1,250.45",
        "$99.00",
        " 500 ",
        "(125.50)"
    ]
    '''for cur in currency_list:
        result=currency_validation(cur)
        print(result)'''

    print(currency_validation("500"))

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-8773352647210965>, line 60
     50 currency_list=[
     51     "1,250.45",
     52     "$99.00",
     53     " 500 ",
     54     "(125.50)"
     55 ]
     56 '''for cur in currency_list:
     57     result=currency_validation(cur)
     58     print(result)'''
---> 60 print(currency_validation("500"))

File <command-8773352647210965>, line 21, in currency_validation(currency)
     19 #incoming value has supported currency
     20 currency_symbol=CURRENCY_SYMBOL_PATTERN.findall(cleaned_currency)
---> 21 if currency_symbol not in SUPPORTED_CURRENCIES and \
     22      (currency_symbol not in('(' , ')')):
     23     raise ValueError(f"unsupported currency. supported currencies are {SUPPORTED_CURRENCIES}")
     24 #incoming value has commas then remove them

TypeError: unhashable type: 'list'

In [0]:
#Parse Monetary Values — Build a function that converts incoming monetary values such as "1,250.45", "$99.00", " 500 ", integers, floats, empty strings, and malformed values into a consistent numeric representation.
import re
from typing import Final, TypeAlias
import math
from decimal import Decimal, InvalidOperation

_SUPPORTED_CURRENCIES:Final[tuple[str,...]]=('USD', '$')
Monetory_Input:TypeAlias=str | int |Decimal | float
CURRENCY_PATTERN:Final[re.Pattern[str]]=re.compile(r"(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d{1,2})?$")

def parse_monetory_value(value:Monetory_Input)->Decimal:
    """Parse a US-formatted monetary input into a Decimal.

    Supports optional '$' or 'USD' prefixes, comma grouping, a leading minus,
    and accounting negatives enclosed in parentheses.

    Raises:
        TypeError: If value is an unsupported type.
        ValueError: If value is missing, malformed, non-finite, or unsupported.
    """
    if isinstance(value, bool):
        raise TypeError(f"monetory value cannot be boolean")     
    #finite value check
    if isinstance(value, Decimal):
        if not value.is_finite():
            raise ValueError(f"monetory value must be finite")
        return value
    if isinstance(value, int):
        return Decimal(value)
    if isinstance(value, float):
        if not math.isfinite(value):
            raise ValueError(f"monetory value must be finite")
        return Decimal(str(value))
    if not isinstance(value, str):
        raise TypeError(f"Monetary value must be str, int, float, or Decimal;"
                        f"received {type(value).__name__}")
    normalized_value=value.strip()
    if not normalized_value:
        raise ValueError(f"monetoy value cannot be empty")

    is_accounting_negative=normalized_value.startswith("(") or normalized_value.endswith(")")
    if is_accounting_negative:
        if not (normalized_value.startswith("(") and normalized_value.endswith(")")):
            raise ValueError(
                f"Malformed accounting negative monetary value: {value!r}.")
        normalized_value=normalized_value[1:-1].strip()
    for prefix in _SUPPORTED_CURRENCIES:
        if normalized_value.startswith(prefix):
            normalized_value=normalized_value[len(prefix):].strip()
            break
    if normalized_value.startswith(("$", "USD")):
        raise ValueError(f"malformed or repeated currencies {value!r}")
    is_negative=normalized_value.startswith('-')
    if is_negative:
        normalized_value=normalized_value[1:]
    if not CURRENCY_PATTERN.fullmatch(normalized_value):
        raise ValueError(f"invalid US currency format{value!r}")
    try:
        amount=Decimal(normalized_value.replace(",",""))
    except InvalidOperation as exc:
        raise ValueError(f"unable to parse monetory value:{value!r}") from exc

    if is_negative or is_accounting_negative:
        amount=-amount

    return amount

if __name__=="__main__":
    currency_list=[
        "1,250.45",
        "$99.00",
        " 500 ",
        "(125.50)"
    ]
    for cur in currency_list:
        result=parse_monetory_value(cur)
        print(result)


1250.45
99.00
500
-125.50
